# 02 · Baseline AutoML → BoW + SVM (binário e multi-rótulo)

- **Origem:** `experiments/automl.ipynb`
- **Faz:** reproduz o *baseline* do artigo — classificação binária (Tab. 7–11) e multi-rótulo — com **Bag-of-Words + SVM**. O `auto-sklearn` original (Linux-only) fica **documentado** numa célula à parte.
- **Entradas:** `experiments/data/1annotator.zip` (split binário) e `ToLD-BR.csv` (multi-rótulo).
- **Saídas:** `results/binary_baseline_bow_svm.json`, `results/multilabel_baseline_bow_svm.json`.

> **Proveniência dos JSONs:** o notebook `03_classification.ipynb` também executa os dois baselines (protocolo do `classification.ipynb` do autor: binário treinado em train+validation = 18.900; multi-rótulo 90/10 com teste = 2.100) e grava **os mesmos arquivos JSON**. Na ordem `01 → 02 → 03`, os JSONs persistidos — e os números do `REPORT.md` e do artigo de reprodução (macro-F1 0.730; Hamming 0.069 / AP 0.209) — são os do `03`. Os resultados deste notebook (0.728; Hamming 0.065 / AP 0.199, protocolos do `automl.ipynb`) ficam registrados nas saídas das células abaixo.

> **Diff mínimo.** Cada alteração abaixo é marcada **[T]** tecnologia descontinuada, **[B]** bug ou **[A]** fidelidade ao artigo — catálogo completo em [`CHANGES.md`](CHANGES.md).

In [1]:
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "ToLD-BR.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_ZIP  = ROOT / "experiments" / "data" / "1annotator.zip"
MAIN_CSV  = ROOT / "ToLD-BR.csv"
ALPHA_CSV = ROOT / "ToLD-BR_alpha.csv"
RESULTS   = ROOT / "reproduction" / "results"
FIGURES   = RESULTS / "figures"
RESULTS.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)

**Mudança #1 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | removidos os installs do auto-sklearn |
| **Antes** | `!sudo apt-get install build-essential swig` + `!pip install auto-sklearn` + `!pip install liac-arff` |
| **Depois** | célula `%pip install scikit-learn pandas matplotlib` (comentada) — o baseline roda em scikit-learn puro |
| **Motivo** | auto-sklearn 0.x (2020) não instala/roda em ambientes atuais (Windows/Python 3.11+) e exige toolchain C; o baseline foi reimplementado em scikit-learn puro |
| **Impacto** | nenhum no resultado (só dependências); o método auto-sklearn original está documentado em célula própria e replicável via WSL/Docker |

In [ ]:
# (T) Dependencias enxutas: o baseline roda em scikit-learn puro (sem auto-sklearn).
# Descomente se faltar algo no ambiente:
# %pip install scikit-learn pandas matplotlib

**Mudança #2 · [T] tecnologia · [A] fidelidade**

| | |
|:--|:--|
| **O quê** | carga binária a partir do ZIP e classificador trocado |
| **Antes** | `pd.read_csv("ptbr_train_1annotator.csv")` (arquivos soltos no Colab) + `autosklearn.classification.AutoSklearnClassifier().fit(...)` |
| **Depois** | `load_binary_split()` lê o split do `DATA_ZIP` + `CountVectorizer` + `SVC().fit(...)` |
| **Motivo** | (T) os CSVs do split binário agora vivem dentro de `experiments/data/1annotator.zip` e o auto-sklearn não roda no ambiente; (A) substituímos o AutoML por BoW+SVM (CountVectorizer + SVC), família de classificador equivalente, preservando a sequência carga→vetorização→fit→predict do autor |
| **Impacto** | macro-F1 0.728 neste notebook (treino só em train = 16.800; vs 0.733 do auto-sklearn reproduzido / 0.74 do artigo). O valor publicado (0.730) vem do baseline do `03` (treino em train+validation = 18.900), que sobrescreve o JSON — ver nota de proveniência no topo |

## Metodo original (auto-sklearn) — preservado para referencia

O autor usava **auto-sklearn** como AutoML para o baseline. Numeros do notebook original / artigo / REPORT.md:

- **Binario:** auto-sklearn best validation score = **0.7546** (accuracy; `0.753968` no notebook original). No teste: artigo reporta **macro-F1 = 0.74**; a reproducao do auto-sklearn em Ubuntu/Docker deu **macro-F1 = 0.733** (REPORT.md).
- **Multi-rotulo:** auto-sklearn best validation score = **0.3311** (f1_macro). No teste do notebook original: **Hamming loss = 0.0838**, **average precision = 0.2019**.

Aqui substituimos o auto-sklearn por **BoW + SVM** (CountVectorizer + SVC), familia de classificador equivalente a que o auto-sklearn tipicamente seleciona para texto esparso e a mesma do baseline do artigo. Esperado nas saidas deste notebook: macro-F1 ~ **0.728** (binario, treino so em train = 16.800; vs 0.74 do artigo); multi-rotulo Hamming ~ **0.065**, AP ~ **0.199** (teste = 1.050, split 0.9/0.5 do autor). Os numeros publicados no REPORT.md e no artigo de reproducao (macro-F1 0.730; Hamming 0.069 / AP 0.209) vem do baseline do notebook `03`, que sobrescreve os JSONs — ver nota de proveniencia no topo.

**Como rodar o auto-sklearn original (opcional, via WSL/Docker, NAO necessario para esta reproducao):**

```bash
# WSL Ubuntu 22.04 ou container Linux (auto-sklearn e Linux-only):
sudo apt-get update && sudo apt-get install -y build-essential swig
python3 -m venv .autosklearn && source .autosklearn/bin/activate
pip install "auto-sklearn==0.14.7" "scikit-learn<0.25" liac-arff pandas
# depois reexecutar as celulas originais (cell-1 e cell-11 de experiments/automl.ipynb),
# substituindo CountVectorizer()/SVC() por autosklearn.classification.AutoSklearnClassifier().
```

In [2]:
import io
import zipfile

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC


def load_binary_split(split):
    """split in {train, validation, test}. Le do ZIP. Retorna [text, toxic]."""
    with zipfile.ZipFile(DATA_ZIP) as z:
        raw = z.read(f"1annotator/ptbr_{split}_1annotator.csv")
    df = pd.read_csv(io.BytesIO(raw))
    return df[["text", "toxic"]].dropna().reset_index(drop=True)


train = load_binary_split("train")
validation = load_binary_split("validation")
test = load_binary_split("test")
print("tamanhos:", len(train), len(validation), len(test))

# (A) BoW + SVM no lugar do auto-sklearn.
bow = CountVectorizer()
X_train = bow.fit_transform(train["text"])
y_train = list(train["toxic"])

X_validation = bow.transform(validation["text"])
y_validation = list(validation["toxic"])

X_test = bow.transform(test["text"])
y_test = list(test["toxic"])

clf = SVC()
clf.fit(X_train, y_train)

tamanhos: 16800 2100 2100


,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


**Mudança #2b · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | `automl.sprint_statistics()` removido |
| **Antes** | `print(automl.sprint_statistics())` (resumo de runs do AutoML) |
| **Depois** | `print(clf)` — o `SVC` não tem resumo de runs |
| **Motivo** | `SVC` não tem esse método; o resumo do auto-sklearn está na célula "Método original" |
| **Impacto** | nenhum no resultado. (Esta anotação reusa o número #2 com sufixo `b` por ser a continuação da mesma migração de tecnologia; também cobre a célula 3, em que `automl.predict` vira `clf.predict`.) |

In [3]:
# (T) sprint_statistics() era do auto-sklearn; SVC nao possui resumo de runs.
print("Classificador binario:", clf)

Classificador binario: SVC()


In [4]:
y_pred = clf.predict(X_test)

In [5]:
import json

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# (A) Persistir metricas-chave para conferencia automatizada.
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
binary_result = {
    "accuracy": report["accuracy"],
    "macro_f1": report["macro avg"]["f1-score"],
    "weighted_f1": report["weighted avg"]["f1-score"],
    "report": report,
    "confusion_matrix": cm.tolist(),
}
with open(RESULTS / "binary_baseline_bow_svm.json", "w", encoding="utf-8") as f:
    json.dump(binary_result, f, indent=2, ensure_ascii=False)
print("macro_f1 =", round(binary_result["macro_f1"], 4))

              precision    recall  f1-score   support

           0       0.73      0.80      0.76      1128
           1       0.74      0.65      0.69       972

    accuracy                           0.73      2100
   macro avg       0.73      0.73      0.73      2100
weighted avg       0.73      0.73      0.73      2100

[[908 220]
 [342 630]]
macro_f1 = 0.7276


# Multi-Label Classification

**Mudança #3 · [T] tecnologia · [A] fidelidade**

| | |
|:--|:--|
| **O quê** | carga multi-rótulo sem gdown e sem auto-sklearn |
| **Antes** | `import autosklearn...` + `import gdown` + `gdown.download(<google-drive-url>, "ToLD-BR.csv")` + `pd.read_csv("ToLD-BR.csv")` com `data.iloc[:, 1:].apply(lambda x: [int(bool(v)) for v in x])` |
| **Depois** | lê `MAIN_CSV` e binariza com `(data[LABELS] > 0)`, mantendo `train_test_split` 0.9/0.5 `random_state=42` |
| **Motivo** | (T) `gdown`/Google Drive e `autosklearn` não se aplicam ao ambiente local; lemos `MAIN_CSV`, já presente no repo. (A) binarizamos `>0` (equivalente a `int(bool(v))`), mantendo o `train_test_split` 0.9/0.5 com `random_state=42` do autor |
| **Impacto** | split idêntico ao original (18900/1050/1050) |

In [6]:
from sklearn.metrics import hamming_loss, average_precision_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
import numpy as np

SEED = 42
LABELS = ["homophobia", "obscene", "insult", "racism", "misogyny", "xenophobia"]

# (T) Sem gdown: lemos o CSV ja versionado no repo via MAIN_CSV.
data = pd.read_csv(MAIN_CSV)
# (A) Binarizacao >0 (equivale a int(bool(v)) do original), restrita as 6 colunas de rotulo.
data[LABELS] = (data[LABELS] > 0).astype(int)

train, test = train_test_split(data, train_size=0.9, random_state=SEED)
test, validation = train_test_split(test, train_size=0.5, random_state=SEED)
print("tamanhos multi-rotulo:", len(train), len(validation), len(test))

tamanhos multi-rotulo: 18900 1050 1050


In [ ]:
# (T) gdown.download(...) removido: o CSV vem de MAIN_CSV (ver celula consolidada acima).

In [ ]:
# (T,A) Carga/binarizacao/split movidos para a celula consolidada acima (MUDANCA #3).

In [7]:
np.array(train[LABELS])

array([[0, 0, 1, 0, 0, 0],
       [0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       ...,
       [0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0]], shape=(18900, 6))

In [8]:
bow = CountVectorizer()
bow.fit(train["text"])

X_train = bow.transform(train["text"])
y_train = np.array(train[LABELS])

X_validation = bow.transform(validation["text"])
y_validation = np.array(validation[LABELS])

X_test = bow.transform(test["text"])
y_test = np.array(test[LABELS])

**Mudança #4 · [B] bug · [A] fidelidade**

| | |
|:--|:--|
| **O quê** | classificador multi-rótulo corrigido e trocado |
| **Antes** | `autosklearn.classification.AutoSklearnClassifiervb()` (método INEXISTENTE — typo de `AutoSklearnClassifier`; jamais rodaria) seguido de `.fit(X, y, X_test, y_test)` com `y` 2D |
| **Depois** | Binary Relevance — um `SVC()` por rótulo (laço em `LABELS`, treinado coluna a coluna) |
| **Motivo** | (B) corrigir o método inexistente; (A) implementar o multi-rótulo como Binary Relevance — um SVC por rótulo, já que SVC não aceita alvo multi-rótulo diretamente |
| **Impacto** | Hamming loss ~ 0.065 e average precision ~ 0.199 neste notebook (teste = 1.050, split 0.9/0.5 do autor; vs 0.0838 / 0.2019 do auto-sklearn original). Os valores publicados (0.069 / 0.209) vêm do baseline do `03` (teste = 2.100), que sobrescreve o JSON — ver nota de proveniência no topo |

In [9]:
# (B,A) Binary Relevance: um SVC por rotulo. y_train ja e (n, 6); treinamos coluna a coluna.
clf_per_label = []
for j, label in enumerate(LABELS):
    svc = SVC()
    svc.fit(X_train, y_train[:, j])
    clf_per_label.append(svc)
print("treinados", len(clf_per_label), "classificadores (1 por rotulo).")

treinados 6 classificadores (1 por rotulo).


In [10]:
# (T) sprint_statistics() era do auto-sklearn; nao se aplica ao Binary Relevance com SVC.
for label, svc in zip(LABELS, clf_per_label):
    print(label, "->", svc)

homophobia -> SVC()
obscene -> SVC()
insult -> SVC()
racism -> SVC()
misogyny -> SVC()
xenophobia -> SVC()


In [11]:
y_pred = np.zeros((X_test.shape[0], len(LABELS)), dtype=int)
for j, svc in enumerate(clf_per_label):
    y_pred[:, j] = svc.predict(X_test)

In [12]:
hamming_loss(y_test, y_pred)

0.06523809523809523

In [13]:
import json

from sklearn.metrics import multilabel_confusion_matrix

ap = average_precision_score(y_test, y_pred)

# (A) Persistir metricas multi-rotulo para conferencia automatizada.
multilabel_result = {
    "hamming_loss": float(hamming_loss(y_test, y_pred)),
    "average_precision": float(ap),
    "labels": LABELS,
    "per_label_confusion": [c.tolist()
                            for c in multilabel_confusion_matrix(y_test, y_pred)],
}
with open(RESULTS / "multilabel_baseline_bow_svm.json", "w", encoding="utf-8") as f:
    json.dump(multilabel_result, f, indent=2, ensure_ascii=False)
print("hamming_loss =", round(multilabel_result["hamming_loss"], 4),
      "| average_precision =", round(multilabel_result["average_precision"], 4))
ap

hamming_loss = 0.0652 | average_precision = 0.1989


0.19891479508072774